In [1]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/+8-HQ-ipc2-B_unopt_magres.magres"


*This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file*

*Shiva Agarwal*

*Apr 23 2025*

In [2]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
nucleus = 'B'      # nucleus for which parameters are wanted
atom_label = 0      # site for which parameters wanted
Q = 0.0406 #electric quadrupole moment for 11B in barn

In [7]:
for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

11B1 sigma:
 [[90.61313071 -4.06249382  1.66237336]
 [ 2.10242302 84.50681994 -6.70477906]
 [-4.09236683 -2.20696288 76.98119486]]

11B2 sigma:
 [[90.61313071  4.06249382  1.66237336]
 [-2.10242302 84.50681994  6.70477906]
 [-4.09236683  2.20696288 76.98119486]]

11B3 sigma:
 [[90.61313071 -4.06249382  1.66237336]
 [ 2.10242302 84.50681994 -6.70477906]
 [-4.09236683 -2.20696288 76.98119486]]

11B4 sigma:
 [[90.61313071  4.06249382  1.66237336]
 [-2.10242302 84.50681994  6.70477906]
 [-4.09236683  2.20696288 76.98119486]]



In [8]:
for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

11B1 sigma:
 -2.1153804003961834

11B2 sigma:
 -2.1153804003963526

11B3 sigma:
 -2.115380400396204

11B4 sigma:
 -2.115380400396361



In [9]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[-0.273 -1.786  1.062]
 [-1.786 -0.055 -0.251]
 [ 1.062 -0.251  0.327]]

CS Tensor:
 [[90.613 -4.062  1.662]
 [ 2.102 84.507 -6.705]
 [-4.092 -2.207 76.981]]

CS isotropic Tensor:
 [[84.034  0.     0.   ]
 [ 0.    84.034  0.   ]
 [ 0.     0.    84.034]]

CS symmetric Tensor:
 [[90.613 -0.98  -1.215]
 [-0.98  84.507 -4.456]
 [-1.215 -4.456 76.981]]

CS antisymmetric Tensor:
 [[ 0.    -3.082  2.877]
 [ 3.082  0.    -2.249]
 [-2.877  2.249  0.   ]]


In [10]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 2.11378192 -2.11590083  0.00211891] 

 Unsorted Eigenvectors:
 [[-0.65322591  0.74505682 -0.13485644]
 [ 0.59254634  0.61391192  0.52153714]
 [-0.47136477 -0.26077288  0.84250386]] 

Sorted Eigenvalues: 
 [ 0.00211891  2.11378192 -2.11590083] 

Sorted Eigenvectors: 
 [[-0.13485644 -0.65322591  0.74505682]
 [ 0.52153714  0.59254634  0.61391192]
 [ 0.84250386 -0.47136477 -0.26077288]] 


For CS tensor
 Unsorted Eigenvalues:
 [74.76698711 90.7913616  86.54279679] 

 Unsorted Eigenvectors:
 [[-0.09522871 -0.99155659 -0.08801718]
 [-0.4220376   0.12029468 -0.89856188]
 [-0.90156295  0.04842233  0.42992967]] 

Sorted Eigenvalues: 
 [86.54279679 90.7913616  74.76698711] 

Sorted Eigenvectors: 
 [[-0.08801718 -0.99155659 -0.09522871]
 [-0.89856188  0.12029468 -0.4220376 ]
 [ 0.42992967  0.04842233 -0.90156295]] 



In [11]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 0.002118910123382495 2.113781923349208 -2.1159008334727667
CSA Tensor Components δyy, δxx, δzz: 
 86.54279679463751 90.79136160021373 74.76698710631989


In [12]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 11B1: 
 +--------------+-----------+
| Quantity     |     Value |
+==============+===========+
| CQ (MHz)     | -2.1159   |
+--------------+-----------+
| etaq         |  0.997997 |
+--------------+-----------+
| iso_cs (ppm) | 84.0337   |
+--------------+-----------+
| csa (ppm)    | -9.26673  |
+--------------+-----------+
| etas         |  0.458475 |
+--------------+-----------+


In [13]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_HQ_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/B_HQ_all_results.txt


In [14]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.65322591 -0.13485644  0.74505682]
 [ 0.59254634  0.52153714  0.61391192]
 [-0.47136477  0.84250386 -0.26077288]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-60.773822817669675 105.1159271417697 -39.48784033233394 

Direction cosine csa: 

[[-0.99155659 -0.08801718 -0.09522871]
 [ 0.12029468 -0.89856188 -0.4220376 ]
 [ 0.04842233  0.42992967 -0.90156295]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
83.57394327438962 154.36427681949087 -77.28469996199132 



In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 47.10809689762434 chi: 95.44795545724291 xi: 76.22144914844833 

